# Demo: borrador de email ante incidencia de compras

Flujo: ver las incidencias -> elegir un lote y escribir un comentario -> ver la incidencia, la evaluacion de la IA sobre el historial del proveedor, el prompt (plantilla editable + prompt final) y el borrador generado.

Cada celda se puede volver a ejecutar de forma independiente tras cambiar las variables marcadas con `# <-- editar`.

In [ ]:
import sys
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import INCIDENCIAS_CSV_PATH
from src.historial import calcular_historial_proveedor, formatear_evaluacion_historico
from src.generador import construir_prompt_completo
from src.prompts import PLANTILLA_BORRADOR
from src.validador_borrador import comprobar_numeros_permitidos, sustituir_placeholders
from src.config import LLM_MODEL, LLM_TEMPERATURE, LLM_TOP_K, LLM_TOP_P
from langchain_ollama import ChatOllama

## 1. Incidencias disponibles

In [ ]:
df = pd.read_csv(INCIDENCIAS_CSV_PATH)
df

## 2. Elige un lote y escribe tu comentario (input del comercial)

In [ ]:
LOTE = "L-3811"  # <-- editar: elige un lote de la tabla de arriba
COMENTARIO_COMERCIAL = ""  # <-- editar: tu comentario libre para incluir en el email

fila = df[df["lote"] == LOTE].iloc[0]
proveedor = fila["proveedor"]
lote = fila["lote"]
fecha = fila["tiempo_compra"]
incidencia_actual = fila["incidencia"]

fila.to_frame().T

## 3. Evaluacion de la IA sobre el historial del proveedor

In [ ]:
historial = calcular_historial_proveedor(df, proveedor, fecha_referencia=date.today().isoformat())
evaluacion_historico = formatear_evaluacion_historico(historial)

print(evaluacion_historico)
historial

## 4. Tono y objetivo deseados

In [ ]:
TONO = "firme-pero-cordial"  # <-- editar: formal / firme-pero-cordial / conciliador
OBJETIVO = "exigir compensacion"  # <-- editar: pedir explicacion / exigir compensacion / avisar / cerrar incidencia

## 5. Plantilla del prompt (editable)

Si necesitas retocar el prompt a mano, edita el texto de `mi_plantilla` (debe conservar los placeholders `{proveedor}`, `{lote}`, `{fecha}`, `{incidencia_actual}`, `{evaluacion_historico}`, `{ejemplos_rag}`, `{tono}`, `{objetivo}`, `{mensaje_libre}`) y vuelve a ejecutar esta celda y las siguientes.

In [ ]:
mi_plantilla = PLANTILLA_BORRADOR  # <-- editar aqui si quieres retocar el prompt
print(mi_plantilla)

## 6. Prompt final ensamblado (con los datos e ejemplos RAG ya insertados)

In [ ]:
prompt_final = construir_prompt_completo(
    proveedor=proveedor,
    lote=lote,
    fecha=fecha,
    incidencia_actual=incidencia_actual,
    historial=historial,
    tono=TONO,
    objetivo=OBJETIVO,
    mensaje_libre=COMENTARIO_COMERCIAL or None,
    plantilla=mi_plantilla,
)
print(prompt_final)

## 7. Borrador generado por el LLM

In [ ]:
llm = ChatOllama(model=LLM_MODEL, temperature=LLM_TEMPERATURE, top_k=LLM_TOP_K, top_p=LLM_TOP_P)
borrador = llm.invoke(prompt_final).content
borrador = borrador.replace("[numero]", lote).replace("[fecha]", fecha)
print(borrador)

## 8. Verificacion anti-alucinacion de cifras (opcional)

In [ ]:
numeros_permitidos = {
    str(historial["num_incidencias_total"]),
    str(historial["num_incidencias_ultimos_6_meses"]),
    str(historial["tasa_reincidencia"]),
    str(historial["gravedad_media"]),
}
discrepancias = comprobar_numeros_permitidos(borrador, numeros_permitidos)
print("Discrepancias:", discrepancias if discrepancias else "ninguna")